In [25]:
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    candidates = []

    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass

    candidates.append(Path(sys.executable).resolve())

    for start in candidates:
        for candidate in (start, *start.parents):
            if (candidate / ".env").exists() or (candidate / "pyproject.toml").exists():
                return candidate

    raise FileNotFoundError("Could not locate the project root containing .env or pyproject.toml")


project_root = find_project_root()
env_path = project_root / ".env"

if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"Loaded environment from {env_path}")
else:
    print(f"No .env file found at {env_path}")


Loaded environment from /Users/anmolgarg/Documents/GenAI/Rag_Tutorial/.env


In [26]:
from langchain_openai import ChatOpenAI



In [27]:
llm  = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [28]:
response = llm.invoke("What is the capital of France?")
response

AIMessage(content='Paris. It’s the capital and largest city of France.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 13, 'total_tokens': 226, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DQvunFJTc01RgBkskMKHLNs2fYCh3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d58d0-10ed-7471-9136-6eb6c86e1648-0', usage_metadata={'input_tokens': 13, 'output_tokens': 213, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}})

In [29]:
response.content

'Paris. It’s the capital and largest city of France.'

RAG IMPLEMENTATION [OWN TEXT DATA]

STEP 1: Preparing document for our text 

In [30]:
from langchain_core.documents import Document

## Create a document with some content
my_text = """Anmol Garg
+91 8650660652 | anmolgarg848@gmail.com
LinkedIn | GitHub | LeetCode
Profile
Data Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data
warehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and
Kafka to develop multiterabyte scalable big data solutions for Fortune 100 Finance and Telecom companies.
Technical Skills
Programming Languages: Python, SQL, Scala, Java
Big Data & Distributed Systems: Apache Spark (PySpark), Hadoop, Hive, YARN, Delta Lake, Apache Hudi,
Streaming Technologies: Kafka, Spark Structured Streaming, Apache Flink
Cloud Platforms: Azure (ADF, ADLS, Synapse, Azure Databricks), AWS (S3, Glue, EMR)
Data Engineering & ETL: Data Modelling, ETL/ELT Pipelines, Delta Lake, DQ
Orchestration & DevOps: Airflow, Azure DevOps, Terraform, CI/CD Pipelines, Kubernetes, Docker
Databases & Caching: Elasticsearch, Redis, Oracle
Experience
EXL Analytics[UK] March 2025 - Present
Data Engineer [Remote]Pune, Maharashtra
• Delivered a project to migrate legacy on-premise processes to the cloud using Big Data technologies (Spark),
reducing processing time by 20%.
• Developed a microservice deployed on Kubernetes, integrating FastAPI to push messages into a Kafka broker
for real-time tracking of processing status (success or failure).
• Partnered with business stakeholders during SIT/UAT phases, ensuring seamless production releases and
eliminating high-severity post-deployment defects.
• Designed and Built ad-hoc file integration pipelines enabling business teams to seamlessly ingest flat files into
the existing data lake architecture, reducing manual onboarding effort and turnaround time.
• Integrated and processed billing data from multiple sources including AWS, Azure, GCP, and Oracle, handling
various file formats like JSON,CSV,Excel and Parquet.
Reliance Jio December 2023 - March 2025
Data Engineer Mumbai, Maharashtra
• Engineered a Hive-driven alert system to identify JioFiber outages, reducing downtime resolution time by 40%,
analyzing 100K+ daily complaints and device disconnections within 60 minutes.
• Optimized overall process performance through Spark performance tuning, improving job run times by 20%
and efficiently managing a 15Terabyte(TB) dataset containing approximately 12 billion records.
• Developed an innovative solution that improved validation for 50,000+ JioFiber WiFi connections per month
based on signal strength, streamlining onboarding processes.
• Designed and implemented advanced scheduling capabilities using Airflow for data pipeline orchestration,
reducing manual intervention time by 80% and streamlining workflow efficiency.
Hexaware Technologies March 2023 - August 2023
Data Analyst, Intern Bhopal, India
• Developed a Spark-based ETL job to automate loading data from server to SQL, replacing the earlier manual
Excel-based process and enabling faster analysis across key engagement KPIs."""

In [31]:
docs = [Document(page_content=my_text, metadata={"source": "my_resume.txt"})]

In [32]:
docs

[Document(metadata={'source': 'my_resume.txt'}, page_content='Anmol Garg\n+91 8650660652 | anmolgarg848@gmail.com\nLinkedIn | GitHub | LeetCode\nProfile\nData Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data\nwarehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and\nKafka to develop multiterabyte scalable big data solutions for Fortune 100 Finance and Telecom companies.\nTechnical Skills\nProgramming Languages: Python, SQL, Scala, Java\nBig Data & Distributed Systems: Apache Spark (PySpark), Hadoop, Hive, YARN, Delta Lake, Apache Hudi,\nStreaming Technologies: Kafka, Spark Structured Streaming, Apache Flink\nCloud Platforms: Azure (ADF, ADLS, Synapse, Azure Databricks), AWS (S3, Glue, EMR)\nData Engineering & ETL: Data Modelling, ETL/ELT Pipelines, Delta Lake, DQ\nOrchestration & DevOps: Airflow, Azure DevOps, Terraform, CI/CD Pipelines, Kubernetes, Docker\nDatabases & Caching: Elasticsearch, Redis,

Converting document into chunks

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'source': 'my_resume.txt'}, page_content='Anmol Garg\n+91 8650660652 | anmolgarg848@gmail.com\nLinkedIn | GitHub | LeetCode\nProfile\nData Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data\nwarehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and'),
 Document(metadata={'source': 'my_resume.txt'}, page_content='Kafka to develop multiterabyte scalable big data solutions for Fortune 100 Finance and Telecom companies.\nTechnical Skills\nProgramming Languages: Python, SQL, Scala, Java\nBig Data & Distributed Systems: Apache Spark (PySpark), Hadoop, Hive, YARN, Delta Lake, Apache Hudi,'),
 Document(metadata={'source': 'my_resume.txt'}, page_content='Streaming Technologies: Kafka, Spark Structured Streaming, Apache Flink\nCloud Platforms: Azure (ADF, ADLS, Synapse, Azure Databricks), AWS (S3, Glue, EMR)\nData Engineering & ETL: Data Modelling, ETL/ELT Pipelines, Delta Lake, DQ'),
 Docume

Creating Embeddings for the chunks using OpenAI's embedding model

In [34]:
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
ebd  = embeddings_model.embed_query("Who's Anmol Garg?")



In [35]:
print(len(ebd))

1536


Create and store embedding into vector db

In [49]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings_model,persist_directory="/Users/anmolgarg/Documents/GenAI/Rag_Tutorial/.vscode/vector")
vectorstore


In [43]:
context = vectorstore.similarity_search("all chunks related to Anmol Garg")

In [44]:
print(context)

[Document(metadata={'source': 'my_resume.txt'}, page_content='Anmol Garg\n+91 8650660652 | anmolgarg848@gmail.com\nLinkedIn | GitHub | LeetCode\nProfile\nData Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data\nwarehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and'), Document(metadata={'source': 'my_resume.txt'}, page_content='Anmol Garg\n+91 8650660652 | anmolgarg848@gmail.com\nLinkedIn | GitHub | LeetCode\nProfile\nData Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data\nwarehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and'), Document(metadata={'source': 'my_resume.txt'}, page_content='• Integrated and processed billing data from multiple sources including AWS, Azure, GCP, and Oracle, handling\nvarious file formats like JSON,CSV,Excel and Parquet.\nReliance Jio December 2023 - March 2025\nData Engineer Mumba

In [45]:
response = llm.invoke("tell me about anmol garg based on this context: " + str(context))

In [46]:
print(response.content)

Here’s a concise summary of Anmol Garg based on the provided context:

- Profile: Data Engineer with 3+ years of experience in building large-scale data pipelines, ETL processes, and data warehouse solutions.
- Technical skills: Python, SQL, Spark, Docker, Databricks, AWS (and experience with cloud data integration).
- Professional experience:
  - Reliance Jio, Data Engineer, Mumbai, Maharashtra
  - Tenure: December 2023 – March 2025
  - Responsibilities: Integrated and processed billing data from multiple sources including AWS, Azure, GCP, and Oracle; handled data in multiple formats such as JSON, CSV, Excel, and Parquet.
- Contact information: +91 8650660652; anmolgarg848@gmail.com
- Online profiles (placeholders indicated): LinkedIn, GitHub, LeetCode

If you’d like, I can tailor this into a LinkedIn summary, a short resume bio, or extract a clean bullet list for a resume with metrics or achievements.


In [48]:
response = llm.invoke(f"where did Anmol Garg work in 2026? : {context}")
print(response.content)

From the provided document, Anmol Garg worked at Reliance Jio from December 2023 to March 2025 as a Data Engineer. There is no information about 2026 in this snippet, so you can’t determine where he worked in 2026 from this resume alone. If you have additional pages or an updated resume, share them and I can check again.


Using Vector Database



In [50]:
vectorstore_persist = Chroma(persist_directory="/Users/anmolgarg/Documents/GenAI/Rag_Tutorial/.vscode/vector", embedding_function=embeddings_model)

/var/folders/kn/rv0076412y1f6cj_k1y1f7xc0000gn/T/ipykernel_37891/3010084810.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(persist_directory="/Users/anmolgarg/Documents/GenAI/Rag_Tutorial/.vscode/vector", embedding_function=embeddings_model)


In [51]:
vectorstore_persist.similarity_search("Who's Anmol Garg?")

[Document(metadata={'source': 'my_resume.txt'}, page_content='Anmol Garg\n+91 8650660652 | anmolgarg848@gmail.com\nLinkedIn | GitHub | LeetCode\nProfile\nData Engineer with 3+ years of experience building large-scale data pipelines, ETL processes, and data\nwarehouse solutions. Utilized technologies like Python, SQL, Spark, Docker, Databricks, AWS and'),
 Document(metadata={'source': 'my_resume.txt'}, page_content='Orchestration & DevOps: Airflow, Azure DevOps, Terraform, CI/CD Pipelines, Kubernetes, Docker\nDatabases & Caching: Elasticsearch, Redis, Oracle\nExperience\nEXL Analytics[UK] March 2025 - Present\nData Engineer [Remote]Pune, Maharashtra'),
 Document(metadata={'source': 'my_resume.txt'}, page_content='• Developed an innovative solution that improved validation for 50,000+ JioFiber WiFi connections per month\nbased on signal strength, streamlining onboarding processes.\n• Designed and implemented advanced scheduling capabilities using Airflow for data pipeline orchestration,'